In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
DATASET_PATH = "your_dataset.parquet"

df = pd.read_parquet(DATASET_PATH)

ID_COLUMN = "comment_id"
TEXT_COLUMN = "comment_text"

emotion_columns = [
    col for col in df.columns
    if col not in [ID_COLUMN, TEXT_COLUMN]
]

print(df.shape)

In [ ]:
outlier_summary = []

for column in emotion_columns:

    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df[
        (df[column] < lower) |
        (df[column] > upper)
    ]

    outlier_summary.append({
        "Emotion": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower,
        "Upper Bound": upper,
        "Outlier Count": len(outliers),
        "Outlier Percentage": (len(outliers) / len(df)) * 100
    })

summary_df = pd.DataFrame(outlier_summary)

summary_df = summary_df.sort_values(
    by="Outlier Count",
    ascending=False
)

summary_df

In [ ]:
os.makedirs("outlier_analysis", exist_ok=True)

summary_df.to_csv(
    "outlier_analysis/outlier_summary.csv",
    index=False
)

In [ ]:
summary_df.head(10)

In [ ]:
summary_df.tail(10)

In [ ]:
rows = 7
cols = 4

fig, axes = plt.subplots(
    rows,
    cols,
    figsize=(18, 24)
)

axes = axes.flatten()

for i, column in enumerate(emotion_columns):

    axes[i].boxplot(df[column])

    axes[i].set_title(column)
    axes[i].set_ylabel("Score")

for j in range(len(emotion_columns), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()

plt.show()

In [ ]:
fig.savefig(
    "outlier_analysis/emotion_boxplots.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
print(f"Total Features : {len(emotion_columns)}")
print(f"Features with Outliers : {(summary_df['Outlier Count'] > 0).sum()}")

print()

print(summary_df[
    ["Emotion", "Outlier Count", "Outlier Percentage"]
])